# Python Functions, Closures & Decorators

*Parameters · LEGB · Closures · Late-binding · Comprehensions · functools · Real-World Production Scenarios*


---
## Built-ins, Loops & Comprehensions


# Builtins Loops Complete

*Run each cell with **Shift+Enter***

Python Built-in Functions, Loops & Comprehensions — Complete Reference
======================================================================
Covers EVERY Python built-in function with:
  * What it does and WHY it exists
  * Concrete examples
  * Use cases and mental models
  * Gotchas

Also covers:
  PART 1 : All 69 built-in functions (organized by category)
  PART 2 : Loops — for, while, for-else, while-else, nested, break/continue
  PART 3 : Comprehensions — list, dict, set, generator; nested; walrus
  PART 4 : Functional tools — map, filter, sorted, zip, enumerate, reversed
  PART 5 : Type conversion & introspection built-ins
  PART 6 : Numeric & math built-ins
  PART 7 : Object & class built-ins
  PART 8 : I/O & system built-ins

Run: python builtins_loops_complete.py


---
## 🧠 Notebook Mental Model: Functions, Closures & Decorators

> **Think of Python functions as first-class objects that capture their environment.**  
> Every function is just a value. It can be stored, passed, returned, and wrapped — which is the basis of closures and decorators.

### The Big Picture Map

```
BUILT-INS ────── len, range, sorted, zip, enumerate, any, all, …
     │               Python ships 69 of these; know their time complexities
     │
LOOPS ──────────  for / while / for-else / walrus :=
     │               "for x in y" desugers into iter(y) + repeated next()
     │
COMPREHENSIONS ─  [expr for x in it if cond]  ← list
                  {k: v for …}                ← dict
                  {expr for …}                ← set
                  (expr for …)                ← generator (lazy!)
     │
FUNCTIONS ──────  def fn(a, b=1, *args, **kwargs): ...
     │               LEGB scope: Local → Enclosing → Global → Built-in
     │
CLOSURES ───────  inner functions that capture variables from enclosing scope
     │               the captured cell survives even after outer returns
     │
DECORATORS ─────  @decorator = fn = decorator(fn)
                  purely syntactic sugar for higher-order functions
```

### Why / What / How / When at a Glance

| Concept | WHY | WHAT | HOW | WHEN |
|---------|-----|------|-----|------|
| **Built-ins** | Common ops in C-speed | Pre-defined functions in `builtins` module | Always available, no import | Default first choice |
| **Comprehensions** | Readable, fast, scoped | Declarative list/dict/set construction | CPython compiles to `LIST_APPEND` bytecode | Transforming/filtering iterables |
| **Generator expr** | Memory efficient | Lazy sequence | State suspended at each `yield` | Large/infinite sequences |
| **LEGB** | Predictable name resolution | Scope search order | L→E→G→B at runtime | Understanding closures/bugs |
| **Closure** | Capture state without a class | Function + its enclosing scope | The free variable lives in a `__closure__` cell | Factories, callbacks, partial application |
| **Decorator** | Wrap behaviour without modifying | A callable that takes/returns a function | `@deco` = `fn = deco(fn)` before `fn` is bound | Logging, timing, retry, auth |

### Lazy vs Eager Evaluation

```
EAGER  (list comprehension)  →  ALL items computed NOW  →  uses full memory
LAZY   (generator expression) →  items computed ON DEMAND →  O(1) memory

Rule: use lazy when you might not need all items, or items are expensive to compute.
      use eager when you need random access or to iterate multiple times.
```


In [ ]:
from __future__ import annotations

import sys
from typing import Any

## PART 1A — BUILT-IN FUNCTIONS: ITERATION & COLLECTION


### 🧠 Mental Model: Iteration Built-ins

**WHY** — Python's iteration built-ins let you process sequences without writing explicit loops, making code more readable and often faster (C-speed implementation).

**The iteration built-ins mental map:**

```
CONSUME one at a time:        next(iter_obj)
PRODUCE indices + values:     enumerate(it, start=0)
PAIR multiple iterables:      zip(a, b, c)   ← stops at shortest
REVERSE lazily:               reversed(seq)

AGGREGATE:
  sum(it)         — numeric total
  min/max(it)     — extreme value (supports key=)
  any(it)         — True if at least one truthy; short-circuits
  all(it)         — True if ALL truthy; short-circuits

TRANSFORM (lazy):
  map(fn, it)     — apply fn to each element
  filter(fn, it)  — keep elements where fn is truthy

PRODUCE ordered sequence:     sorted(it, key=, reverse=)  ← always new list
```

**Short-circuit importance — use with generators:**
```python
# any() and all() stop as soon as the result is determined
# Using a generator keeps it lazy: no wasted computation
if any(is_valid(x) for x in huge_list):  # stops at first valid
    ...
if all(x > 0 for x in data):            # stops at first non-positive
    ...
```

**`sorted` vs `.sort()` gotcha:**
```
sorted(lst)  → returns NEW sorted list  (lst unchanged)
lst.sort()   → sorts IN-PLACE, returns None  ← common bug: result = lst.sort()
```


In [ ]:
def demo_iteration_builtins() -> None:
    """
    len(obj)      — number of items in a sequence/mapping/set
    iter(obj)     — get an iterator for obj; calls obj.__iter__()
    next(it, def) — advance iterator; returns default if exhausted
    range(n) / range(start,stop,step) — lazy integer sequence
    enumerate(it, start) — (index, value) pairs
    zip(*iters)   — pair items from multiple iterables (stops at shortest)
    reversed(seq) — lazy reverse iterator (needs __reversed__ or __len__+__getitem__)
    sorted(it, key, reverse) — returns new sorted LIST; stable (Timsort)
    min / max     — minimum/maximum with optional key function
    sum(it, start) — numeric total; start defaults to 0
    any(it)       — True if ANY element is truthy; short-circuits
    all(it)       — True if ALL elements are truthy; short-circuits
    map(fn, *its) — lazy apply fn to each item
    filter(fn, it)— lazy keep items where fn is truthy (None → identity)
    """
    # len
    assert len([1,2,3]) == 3
    assert len("hello") == 5
    assert len({1,2}) == 2
    assert len({"a":1}) == 1

    # range — lazy, supports len() and indexing
    r = range(10)
    assert len(r) == 10 and r[3] == 3 and 7 in r
    assert list(range(2,10,3)) == [2,5,8]
    assert list(range(10,0,-2)) == [10,8,6,4,2]

    # enumerate
    words = ["a","b","c"]
    assert list(enumerate(words))       == [(0,"a"),(1,"b"),(2,"c")]
    assert list(enumerate(words,start=1))== [(1,"a"),(2,"b"),(3,"c")]

    # zip — stops at shortest
    assert list(zip([1,2,3],[4,5,6])) == [(1,4),(2,5),(3,6)]
    assert list(zip([1,2,3],[4,5]))   == [(1,4),(2,5)]   # 3 dropped
    # Unzip with *
    pairs = [(1,"a"),(2,"b"),(3,"c")]
    nums, letters = zip(*pairs)
    assert list(nums) == [1,2,3] and list(letters) == ["a","b","c"]

    # reversed
    assert list(reversed([1,2,3])) == [3,2,1]
    assert list(reversed(range(5))) == [4,3,2,1,0]

    # sorted
    data = [3,1,4,1,5,9,2,6]
    assert sorted(data) == [1,1,2,3,4,5,6,9]
    assert sorted(data, reverse=True) == [9,6,5,4,3,2,1,1]
    words2 = ["banana","apple","cherry","date"]
    assert sorted(words2, key=len) == ["date","apple","banana","cherry"]
    # STABLE sort: equal elements maintain original order
    pairs2 = [(1,"b"),(1,"a"),(2,"c")]
    assert sorted(pairs2) == [(1,"a"),(1,"b"),(2,"c")]

    # min / max
    assert min([3,1,4,1,5]) == 1
    assert max([3,1,4,1,5]) == 5
    assert min("apple","banana","cherry")   == "apple"    # lexicographic
    assert max([(1,3),(2,1),(1,5)], key=lambda p:p[1]) == (1,5)
    assert min(3, 7, 2, 9, 0) == 0       # variadic form
    # min/max with default (empty iterable)
    assert min([], default="empty") == "empty"

    # sum
    assert sum([1,2,3,4,5]) == 15
    assert sum(range(101))  == 5050          # Gauss!
    assert sum([[1,2],[3,4]], []) == [1,2,3,4]  # start=[] concatenates lists

    # any / all — short-circuit evaluation
    assert any([False,False,True,False])  # stops at True
    assert all([True,True,True])
    assert not any([])                    # any([]) = False (vacuously)
    assert all([])                        # all([]) = True (vacuously)
    # Use with generator expressions — lazy, efficient
    has_negative = any(x < 0 for x in [1,2,-3,4])
    assert has_negative

    # map — lazy transform
    doubled = list(map(lambda x: x*2, [1,2,3]))
    assert doubled == [2,4,6]
    # map with multiple iterables
    sums = list(map(lambda a,b: a+b, [1,2,3],[10,20,30]))
    assert sums == [11,22,33]

    # filter — lazy selection
    evens = list(filter(lambda x: x%2==0, range(10)))
    assert evens == [0,2,4,6,8]
    # filter(None, it) removes falsy values
    cleaned = list(filter(None, [0,"",None,1,"hello",False]))
    assert cleaned == [1,"hello"]

    print("Part 1A (Iteration built-ins): ✓")

## PART 1B — BUILT-IN FUNCTIONS: TYPE CONVERSION

In [ ]:
def demo_type_conversion() -> None:
    """
    int(x, base)    — convert to int; base for string parsing
    float(x)        — convert to float
    complex(r, i)   — construct complex number
    str(x)          — human-readable string (calls __str__)
    repr(x)         — developer string (calls __repr__)
    bytes(x, enc)   — convert to bytes
    bytearray(x)    — mutable bytes
    bool(x)         — truthiness; calls __bool__ then __len__
    list(it)        — materialise iterable into list
    tuple(it)       — materialise into tuple
    set(it)         — materialise into set (dedup)
    frozenset(it)   — immutable set
    dict(**kwargs) / dict(mapping) / dict(pairs) — construct dict
    """
    # int
    assert int("42")     == 42
    assert int("0b1010", 2) == 10     # binary string
    assert int("ff", 16) == 255       # hex string
    assert int(3.9)      == 3         # truncates (floor toward zero)
    assert int(-3.9)     == -3        # NOT -4 — truncates toward zero!
    assert int(True)     == 1
    assert int(False)    == 0

    # float
    assert float("3.14") == 3.14
    assert float("inf")  == float("inf")
    assert float("nan") != float("nan")   # NaN is never equal to itself

    # str / repr
    assert str(42)      == "42"
    assert str([1,2,3]) == "[1, 2, 3]"
    assert repr("hello") == "'hello'"     # includes quotes
    assert repr([1,2])   == "[1, 2]"

    # bytes / bytearray
    b = bytes("hello","utf-8")
    assert b == b"hello"
    ba = bytearray("hello","utf-8")
    ba[0] = 72
    assert ba == b"Hello"

    # bool
    assert bool(0) is False
    assert bool("") is False
    assert bool([]) is False
    assert bool(None) is False
    assert bool(42) is True
    assert bool("x") is True

    # list / tuple / set / frozenset
    assert list(range(3))        == [0,1,2]
    assert tuple(range(3))       == (0,1,2)
    assert set([1,2,2,3])        == {1,2,3}
    assert frozenset([1,2,2,3])  == frozenset({1,2,3})

    # dict constructors
    d1 = dict(a=1, b=2)
    d2 = dict([("a",1),("b",2)])
    d3 = dict({"a":1,"b":2})
    assert d1 == d2 == d3

    print("Part 1B (Type conversion): ✓")

## PART 1C — BUILT-IN FUNCTIONS: NUMERIC & MATH

In [ ]:
def demo_numeric_builtins() -> None:
    """
    abs(x)         — absolute value; calls __abs__
    round(x, n)    — round to n decimal places (banker's rounding!)
    divmod(a, b)   — (quotient, remainder) in one call
    pow(x, y, mod) — x**y, optionally mod; efficient modular exponentiation
    hash(x)        — hash code; equal objects must have equal hashes
    id(x)          — identity (CPython: memory address)
    hex(n)         — integer to hex string: "0xff"
    oct(n)         — integer to octal string: "0o17"
    bin(n)         — integer to binary string: "0b1010"
    ord(c)         — character to Unicode code point
    chr(n)         — Unicode code point to character
    """
    # abs
    assert abs(-5) == 5
    assert abs(-3.14) == 3.14
    assert abs(3+4j) == 5.0   # complex magnitude

    # round — BANKER'S ROUNDING (round half to even)!
    assert round(2.5) == 2    # rounds to nearest even (2)
    assert round(3.5) == 4    # rounds to nearest even (4)
    assert round(2.675, 2) != 2.68  # float repr issue
    assert round(1.234567, 2) == 1.23

    # divmod
    assert divmod(17, 5) == (3, 2)   # 17 = 3*5 + 2
    assert divmod(-17, 5) == (-4, 3) # floor division semantics

    # pow
    assert pow(2, 10) == 1024
    assert pow(2, 10, 1000) == 24    # (2**10) % 1000 — efficient for crypto
    assert pow(2, -1) == 0.5

    # hash
    assert hash(42) == hash(42)
    assert hash("hello") == hash("hello")
    # Equal objects must hash equal
    assert hash(1) == hash(1.0) == hash(True)   # 1 == 1.0 == True

    # id
    x = "hello"
    assert id(x) == id(x)    # same object
    a, b = [], []
    assert id(a) != id(b)    # different objects

    # Number base conversions
    assert hex(255) == "0xff"
    assert oct(8)   == "0o10"
    assert bin(10)  == "0b1010"
    assert int("0xff",16) == 255

    # ord / chr
    assert ord("A") == 65
    assert chr(65)  == "A"
    assert ord("€") == 8364
    assert "".join(chr(i) for i in range(65,91)) == "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

    print("Part 1C (Numeric built-ins): ✓")

## PART 1D — BUILT-IN FUNCTIONS: OBJECT & CLASS INTROSPECTION

In [ ]:
def demo_introspection_builtins() -> None:
    """
    type(obj)             — get the type of obj
    isinstance(obj, cls)  — checks type including base classes
    issubclass(cls, base) — checks class hierarchy
    hasattr(obj, name)    — True if obj has attribute name
    getattr(obj, name, default) — get attribute with optional default
    setattr(obj, name, val)     — set attribute
    delattr(obj, name)          — delete attribute
    dir(obj)              — list of attribute names
    vars(obj)             — __dict__ of obj
    callable(obj)         — True if obj has __call__
    super()               — proxy to MRO parent
    object()              — base class of all classes
    property()            — create property descriptor
    classmethod()         — create classmethod descriptor
    staticmethod()        — create staticmethod descriptor
    """
    class Animal:
        def __init__(self, name):
            self.name = name
        def speak(self): return "..."

    class Dog(Animal):
        def speak(self): return "Woof"

    d = Dog("Rex")

    # type
    assert type(d) is Dog
    assert type(d) is not Animal
    assert type([]) is list
    assert type(type) is type      # type is its own metaclass!

    # isinstance — checks full MRO
    assert isinstance(d, Dog)
    assert isinstance(d, Animal)   # Dog IS-A Animal
    assert isinstance(d, object)   # everything IS-A object
    assert isinstance(42, (int, float))  # accepts tuple of types

    # issubclass
    assert issubclass(Dog, Animal)
    assert issubclass(bool, int)   # bool is a subclass of int!
    assert not issubclass(Animal, Dog)

    # hasattr / getattr / setattr / delattr
    assert hasattr(d, "name")
    assert not hasattr(d, "age")
    assert getattr(d, "name") == "Rex"
    assert getattr(d, "age", 0) == 0    # default if missing
    setattr(d, "age", 5)
    assert d.age == 5
    delattr(d, "age")
    assert not hasattr(d, "age")

    # callable
    assert callable(Dog)             # classes are callable
    assert callable(len)             # functions are callable
    assert not callable(42)

    class CallMe:
        def __call__(self): return "called"

    assert callable(CallMe())        # instances with __call__ are callable

    # vars — returns __dict__
    d2 = Dog("Spot")
    d2.age = 3
    assert vars(d2) == {"name":"Spot","age":3}

    # dir — includes inherited attributes
    attrs = dir(d)
    assert "speak" in attrs and "name" in attrs and "__class__" in attrs

    print("Part 1D (Introspection built-ins): ✓")

## PART 1E — REMAINING BUILT-IN FUNCTIONS

In [ ]:
def demo_remaining_builtins() -> None:
    """
    open(file, mode, encoding) — file object
    print(*args, sep, end, file, flush)
    input(prompt)    — read a line from stdin (not demonstrated)
    format(val, spec)— format a value using a format spec
    ascii(x)         — repr with non-ASCII escaped
    chr(n)/ord(c)    — already covered
    compile()        — compile source to code object (advanced)
    eval(expr)       — evaluate a Python expression string
    exec(code)       — execute Python code string/code object
    globals()        — current module's global namespace dict
    locals()         — current local namespace dict
    breakpoint()     — invokes pdb.set_trace() (debugging)
    __import__(name) — low-level import
    memoryview(buf)  — zero-copy buffer protocol
    slice(stop) / slice(start,stop,step) — slice object
    """
    # format
    assert format(3.14159, ".2f")  == "3.14"
    assert format(42, "08b")       == "00101010"   # 8-digit binary
    assert format(1000000, ",")    == "1,000,000"
    assert len(format("hello", "^20")) == 20          # centered in 20 chars

    # ascii — repr with non-ASCII chars escaped
    assert ascii("café") == "'caf\\xe9'"

    # eval — evaluate expression (USE WITH CAUTION — security risk!)
    result = eval("2 + 2")
    assert result == 4
    # NEVER eval() untrusted user input — it executes arbitrary code!

    # globals / locals
    g = globals()
    assert "__name__" in g and "__file__" in g

    def scope_demo():
        x, y = 10, 20
        l = locals()
        assert "x" in l and l["x"] == 10
    scope_demo()

    # slice
    s = slice(2, 8, 2)           # equivalent to [2:8:2]
    data = list(range(10))
    assert data[s] == [2, 4, 6]
    assert data[::2] == data[slice(None,None,2)]

    # memoryview — zero-copy access to buffer protocol
    b = bytearray(b"hello world")
    view = memoryview(b)
    view[0:5] = b"HELLO"
    assert bytes(b) == b"HELLO world"   # modified in-place, zero copy

    # hash of common types
    assert hash(frozenset([1,2,3])) == hash(frozenset([3,1,2]))  # order-independent
    try:
        hash([1,2,3])  # TypeError: unhashable type: 'list'
        assert False
    except TypeError:
        pass

    print("Part 1E (Remaining built-ins): ✓")

## PART 2 — LOOPS


### 🧠 Mental Model: Loops

**WHY** — Python's loop constructs go beyond simple iteration: `for-else` eliminates flag variables; `walrus :=` removes redundant re-computation; `break/continue` provide fine-grained control.

**Mental model — `for` loop desugars to:**
```python
# for x in iterable:  body
_iter = iter(iterable)       # calls iterable.__iter__()
while True:
    try:
        x = next(_iter)      # calls _iter.__next__()
    except StopIteration:
        break                # loop exhausted — run the else clause
    body
```

**`for-else` — eliminates the "found" flag pattern:**
```python
# Without for-else (verbose, flag variable)
found = False
for item in collection:
    if predicate(item):
        found = True
        process(item)
        break
if not found:
    handle_not_found()

# With for-else (clean, no flag)
for item in collection:
    if predicate(item):
        process(item)
        break
else:
    handle_not_found()   # only runs if loop completed without break
```

**`walrus :=` — assign and test in one step:**
```python
# Classic while — compute expression twice (bug-prone if it has side effects)
line = file.readline()
while line:
    process(line)
    line = file.readline()

# Walrus — compute once, assign, test
while line := file.readline():
    process(line)
```

**WHEN to choose each loop type:**
| Scenario | Best tool |
|----------|-----------|
| Iterate over known sequence | `for x in seq` |
| Indexed iteration | `for i, x in enumerate(seq)` |
| Parallel sequences | `for a, b in zip(seq1, seq2)` |
| Unknown number of iterations | `while condition` |
| Search with "not found" branch | `for … else` |
| Read-until-empty pattern | `while chunk := f.read(n)` |


In [ ]:
def demo_loops() -> None:
    """
    for loop    — iterate over any iterable; for-else
    while loop  — condition-controlled; while-else
    break       — exit the loop immediately (skips else)
    continue    — skip to next iteration
    pass        — empty body placeholder
    nested loops — matrix traversal, combinations
    walrus :=   — assign-and-test in while conditions
    """
    # Basic for
    total = 0
    for i in range(1, 6):
        total += i
    assert total == 15

    # for over different iterables
    chars  = [c for c in "hello"]   # str is iterable
    keys   = [k for k in {"a":1,"b":2}]  # dict iterates over keys
    items  = [(k,v) for k,v in {"a":1,"b":2}.items()]
    assert chars  == ["h","e","l","l","o"]
    assert set(keys) == {"a","b"}
    assert set(items) == {("a",1),("b",2)}

    # for-else: else runs ONLY if loop finished without break
    def find_prime(nums: list[int]) -> int | None:
        for n in nums:
            if n > 1 and all(n % d != 0 for d in range(2,int(n**0.5)+1)):
                return n   # found — break would be here, else doesn't run
        else:
            return None    # no break → no prime found
    assert find_prime([4,6,8,9,11]) == 11
    assert find_prime([4,6,8,9]) is None

    # while + break + continue
    n, acc = 10, 0
    while n > 0:
        n -= 1
        if n % 3 == 0: continue   # skip multiples of 3
        if acc > 15:   break       # stop if total exceeds 15
        acc += n
    # acc is at most the value that caused the break (≤ 15 + that value)
    assert acc <= 30               # loose bound: always holds

    # while-else
    x = 16
    while x > 1:
        if x % 2 != 0: break
        x //= 2
    else:
        pass   # x reached 1 without break: x was a power of 2

    # walrus := in while — read chunks until empty
    import io
    data = io.StringIO("line1\nline2\nline3\n")
    lines_read = []
    while line := data.readline():   # assign and check truthiness
        lines_read.append(line.strip())
    assert lines_read == ["line1","line2","line3"]

    # Nested loops — matrix
    matrix = [[1,2,3],[4,5,6],[7,8,9]]
    diagonal = [matrix[i][i] for i in range(len(matrix))]
    assert diagonal == [1,5,9]

    # break in nested loop — only breaks INNERMOST loop
    found = None
    for row in range(3):
        for col in range(3):
            if matrix[row][col] == 5:
                found = (row, col); break
        if found: break
    assert found == (1,1)

    # for with enumerate — Pythonic indexed iteration
    fruits = ["apple","banana","cherry"]
    indexed = [(i,f) for i,f in enumerate(fruits, start=1)]
    assert indexed == [(1,"apple"),(2,"banana"),(3,"cherry")]

    # for with zip — parallel iteration
    names   = ["Alice","Bob","Carol"]
    scores  = [85, 92, 78]
    paired  = list(zip(names,scores))
    assert paired == [("Alice",85),("Bob",92),("Carol",78)]
    # GOTCHA: zip stops at shortest; use itertools.zip_longest for padding

    print("Part 2 (Loops): ✓")

## PART 3 — COMPREHENSIONS
Mental model: comprehensions are DECLARATIVE — they say WHAT you want,
  not HOW to compute it. They're usually faster than equivalent loops
  because the CPython interpreter optimises them specially.


### 🧠 Mental Model: Comprehensions

**WHY** — Comprehensions are Python's answer to the boilerplate of building a collection with a loop. They're declarative (say what, not how) and often 20–50 % faster than equivalent loops.

**WHAT — the four forms:**
```
[expr for x in it if cond]     → list  (eager, materialises all at once)
{k: v for x in it if cond}     → dict  (eager)
{expr for x in it if cond}     → set   (eager, deduplicates)
(expr for x in it if cond)     → generator (LAZY — O(1) memory)
```

**HOW — read comprehensions inside-out:**
```
[x * 2  for x in range(10)  if x % 2 == 0]
  ↑           ↑                   ↑
 OUTPUT      SOURCE             FILTER
```

**Nested comprehensions — read the loops top-to-bottom:**
```python
# Equivalent loop order:
for row in matrix:          # outer loop first
    for x in row:           # inner loop second
        flat.append(x)

# Comprehension (same order):
flat = [x for row in matrix for x in row]
```

**Generator vs list — the memory trade-off:**
```
list:      [x*x for x in range(10**6)]   ~8 MB allocated immediately
generator: (x*x for x in range(10**6))   ~200 bytes; computes on demand

Use generator when:
  • Feeding into sum(), any(), all(), max(), min() (never need all at once)
  • Processing data pipelines
  • The iterable is potentially infinite

Use list when:
  • Need random access (indexing)
  • Need to iterate multiple times
  • Need len()
```

**Scope gotcha — Python 3 scopes comprehension variables:**
```python
x = 100
result = [x for x in range(5)]  # x inside is LOCAL
assert x == 100                  # outer x unchanged (unlike Python 2!)
```


In [ ]:
def demo_comprehensions() -> None:
    # ── LIST COMPREHENSION ────────────────────────────────────────────────────
    # [expression for item in iterable if condition]
    squares   = [x*x for x in range(10)]
    even_sq   = [x*x for x in range(10) if x%2==0]
    upper     = [w.upper() for w in "hello world".split()]
    assert squares == [0,1,4,9,16,25,36,49,64,81]
    assert even_sq == [0,4,16,36,64]
    assert upper   == ["HELLO","WORLD"]

    # Nested list comprehension (matrix transpose)
    matrix = [[1,2,3],[4,5,6],[7,8,9]]
    transposed = [[row[i] for row in matrix] for i in range(3)]
    assert transposed == [[1,4,7],[2,5,8],[3,6,9]]

    # Flatten nested list
    flat = [x for row in matrix for x in row]
    assert flat == [1,2,3,4,5,6,7,8,9]

    # ── DICT COMPREHENSION ────────────────────────────────────────────────────
    sq_map  = {x: x*x for x in range(5)}
    inverted = {v: k for k, v in sq_map.items()}
    assert sq_map[3] == 9
    assert inverted[9] == 3

    # Filter dict
    big  = {k: v for k, v in sq_map.items() if v > 5}
    assert big == {3:9, 4:16}

    # ── SET COMPREHENSION ─────────────────────────────────────────────────────
    lengths = {len(w) for w in "the quick brown fox".split()}
    assert lengths == {3,5}        # unique lengths

    # ── GENERATOR EXPRESSION ─────────────────────────────────────────────────
    # Parentheses instead of brackets; LAZY — no list created
    gen = (x*x for x in range(1_000_000))
    # This uses O(1) memory; materialising the list would use ~8 MB
    assert next(gen) == 0 and next(gen) == 1   # computed on demand

    # ── WALRUS := IN COMPREHENSIONS ──────────────────────────────────────────
    # Compute once, filter and use in same expression
    def compute(x): return x * x + 1

    # Without walrus: compute called twice
    naive = [compute(x) for x in range(10) if compute(x) > 10]

    # With walrus: compute called once per item
    smart = [v for x in range(10) if (v := compute(x)) > 10]
    assert naive == smart

    # ── COMPREHENSION SCOPE ───────────────────────────────────────────────────
    # In Python 3, comprehension variables are SCOPED; they don't leak
    x = 100
    _ = [x for x in range(3)]   # this x is LOCAL to the comprehension
    assert x == 100              # outer x unchanged (Python 3 only!)

    print("Part 3 (Comprehensions): ✓")

## PART 4 — SORTING, GROUPING & FUNCTIONAL TRANSFORMS


### 🧠 Mental Model: Sorting, Closures & Decorators

**WHY** — Functions are first-class objects, so they can be passed as `key=` arguments to `sorted()`, pre-filled with `partial()`, or wrapped with decorators to add behaviour without touching original code.

**`sorted()` key function mental model:**
```
sorted(items, key=fn)
 └─ fn is called once per item: key_value = fn(item)
 └─ items are sorted by key_value, NOT by item directly
 └─ Timsort: O(n log n), STABLE (equal items keep original order)

Common key patterns:
  key=len                      # sort strings by length
  key=lambda x: x["age"]      # sort dicts by field
  key=lambda x: (x.y, -x.z)  # multi-key: primary asc, secondary desc
  key=str.lower                # case-insensitive string sort
```

**LEGB Scope & Closures:**
```
def outer(x):
    def inner(y):       ← closure: inner "closes over" x
        return x + y    ← x is a FREE VARIABLE captured from outer's scope
    return inner

add5 = outer(5)         ← outer has returned, but x=5 is captured in add5.__closure__
add5(3) → 8
```

**Late-binding gotcha (most common closure bug):**
```python
# BAD: all lambdas share the SAME i variable (= final value after loop)
funcs = [lambda: i for i in range(3)]
funcs[0]()  → 2, funcs[1]()  → 2, funcs[2]()  → 2  # SURPRISE!

# GOOD: capture current value with a default argument
funcs = [lambda i=i: i for i in range(3)]
funcs[0]()  → 0, funcs[1]()  → 1, funcs[2]()  → 2  # CORRECT
```

**Decorator mental model:**
```
@my_decorator
def greet(name):
    return f"Hello, {name}"

# Exactly equivalent to:
def greet(name):
    return f"Hello, {name}"
greet = my_decorator(greet)

# A well-written decorator:
from functools import wraps
def my_decorator(fn):
    @wraps(fn)          ← preserves fn.__name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        print("before")
        result = fn(*args, **kwargs)
        print("after")
        return result
    return wrapper
```

**WHEN to use each:**
| Tool | When |
|------|------|
| `sorted(key=)` | Any non-default ordering |
| `functools.partial` | Pre-fill args for callbacks/factories |
| Closure | Stateful factory without a class |
| Decorator | Cross-cutting concerns: logging, retry, timing, auth |


In [ ]:
def demo_sorting_functional() -> None:
    from functools import reduce, partial, cmp_to_key

    # ── SORTED / SORT ──────────────────────────────────────────────────────────
    data = [3,1,4,1,5,9,2,6,5,3]
    assert sorted(data)             == [1,1,2,3,3,4,5,5,6,9]
    assert data                     == [3,1,4,1,5,9,2,6,5,3]  # unchanged

    data.sort()                        # in-place
    assert data                     == [1,1,2,3,3,4,5,5,6,9]

    # Key function — sort by arbitrary criterion
    people = [{"name":"Alice","age":30},{"name":"Bob","age":25},{"name":"Carol","age":35}]
    by_age  = sorted(people, key=lambda p: p["age"])
    assert by_age[0]["name"] == "Bob"

    # Multi-key sort (tuples are compared element-by-element)
    students = [("Alice",3,90),("Bob",3,85),("Alice",4,92),("Bob",2,95)]
    sorted_multi = sorted(students, key=lambda s: (s[0], -s[1]))   # name asc, year desc
    assert sorted_multi[0] == ("Alice",4,92)

    # cmp_to_key: convert old-style cmp function to a key function
    def by_last_digit(a, b):
        return (a%10) - (b%10)
    nums = sorted([23,45,12,67,34], key=cmp_to_key(by_last_digit))
    assert nums[0]%10 <= nums[-1]%10

    # ── REDUCE ────────────────────────────────────────────────────────────────
    from functools import reduce
    total   = reduce(lambda acc, x: acc + x, [1,2,3,4,5], 0)
    product = reduce(lambda acc, x: acc * x, [1,2,3,4,5], 1)
    assert total == 15 and product == 120

    # ── PARTIAL ───────────────────────────────────────────────────────────────
    from functools import partial
    def power(base, exp): return base ** exp
    square = partial(power, exp=2)
    cube   = partial(power, exp=3)
    assert square(5) == 25 and cube(3) == 27

    # ── MAP / FILTER (functional vs comprehension) ────────────────────────────
    nums2 = [1,2,3,4,5]
    # map + lambda vs list comprehension — comprehension preferred in Python
    via_map  = list(map(lambda x: x*2, nums2))
    via_comp = [x*2 for x in nums2]
    assert via_map == via_comp

    via_filter  = list(filter(lambda x: x%2==0, nums2))
    via_comp2   = [x for x in nums2 if x%2==0]
    assert via_filter == via_comp2

    print("Part 4 (Sorting/functional): ✓")

## PART 5 — STRING BUILT-INS & FORMATTING

In [ ]:
def demo_string_builtins() -> None:
    """All major str methods and formatting techniques."""
    s = "  Hello, World!  "

    # Whitespace
    assert s.strip()  == "Hello, World!"
    assert s.lstrip() == "Hello, World!  "
    assert s.rstrip() == "  Hello, World!"
    assert "a  b  c".split()    == ["a","b","c"]
    assert "a,b,,c".split(",")  == ["a","b","","c"]
    assert "a,b,,c".split(",",2) == ["a","b",",c"]
    assert "\n".join(["a","b","c"]) == "a\nb\nc"

    # Case
    assert "hello".upper()      == "HELLO"
    assert "HELLO".lower()      == "hello"
    assert "hello world".title()== "Hello World"
    assert "hello World".swapcase()== "HELLO wORLD"

    # Search / Replace
    assert "hello".find("ll")   == 2   # -1 if not found
    assert "hello".index("ll")  == 2   # ValueError if not found
    assert "hello".rfind("l")   == 3   # right-most
    assert "hello".count("l")   == 2
    assert "hello".replace("l","L") == "heLLo"
    assert "hello".replace("l","L",1) == "heLlo"  # max 1 replacement

    # Tests
    assert "hello".startswith("he")
    assert "hello".endswith("lo")
    assert "  ".isspace()
    assert "abc123".isalnum()
    assert "abc".isalpha()
    assert "123".isdigit()
    assert "123".isnumeric()
    assert "HELLO".isupper()
    assert "hello".islower()

    # Padding / alignment
    assert "hi".center(10)   == "    hi    "
    assert "hi".ljust(10,"_")== "hi________"
    assert "hi".rjust(10,"_")== "________hi"
    assert "42".zfill(6)     == "000042"

    # Partition
    assert "a=b=c".partition("=")  == ("a","=","b=c")    # first occurrence
    assert "a=b=c".rpartition("=") == ("a=b","=","c")    # last occurrence

    # Format — three approaches
    assert "Hello, {}!".format("World") == "Hello, World!"
    assert "Hello, {name}!".format(name="World") == "Hello, World!"
    name = "World"
    assert f"Hello, {name}!" == "Hello, World!"           # f-string (fastest)

    # f-string formatting specs
    pi = 3.14159
    assert f"{pi:.2f}" == "3.14"
    assert f"{pi:10.3f}" == "     3.142"
    assert f"{42:08b}" == "00101010"       # binary
    assert f"{'left':<10}" == "left      "
    assert f"{'right':>10}" == "     right"
    assert f"{'center':^10}" == "  center  "

    # encode / decode
    encoded = "hello".encode("utf-8")
    assert encoded == b"hello"
    assert encoded.decode("utf-8") == "hello"

    # maketrans / translate — character substitution
    trans = str.maketrans("aeiou","*****")
    assert "hello world".translate(trans) == "h*ll* w*rld"

    print("Part 5 (String built-ins): ✓")

## PART 6 — I/O BUILT-INS

In [ ]:
def demo_io_builtins() -> None:
    """
    open(file, mode, encoding, errors, buffering)
    Modes: 'r','w','a','x','b','t','+' and combinations
    print(*args, sep, end, file, flush)
    """
    import tempfile, os

    # Writing a file
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt",
                                     encoding="utf-8", delete=False) as f:
        path = f.name
        print("line1", "line2", sep="\n", file=f)  # print to file
        f.write("line3\n")

    # Reading — multiple modes
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    assert [l.strip() for l in lines] == ["line1","line2","line3"]

    # Reading lazily (line by line — O(1) memory)
    with open(path, encoding="utf-8") as f:
        first = next(f).strip()         # iterate the file object directly
    assert first == "line1"

    # Binary mode
    with open(path, "rb") as f:
        raw = f.read()
    assert isinstance(raw, bytes)

    # seek / tell
    with open(path, encoding="utf-8") as f:
        f.seek(0)
        pos = f.tell()
        assert pos == 0
        f.read(5)
        assert f.tell() == 5

    # append mode
    with open(path, "a", encoding="utf-8") as f:
        f.write("line4\n")
    with open(path, encoding="utf-8") as f:
        content = f.read()
    assert "line4" in content

    # x mode — exclusive creation (fails if file exists)
    try:
        open(path, "x").close()   # file already exists → FileExistsError
    except FileExistsError:
        pass

    os.unlink(path)

    # print() customisation
    import io
    buf = io.StringIO()
    print("a","b","c", sep="|", end="!\n", file=buf)
    assert buf.getvalue() == "a|b|c!\n"

    print("Part 6 (I/O built-ins): ✓")

## MAIN

In [ ]:
def main() -> None:
    print("=" * 70)
    print("BUILT-INS / LOOPS / COMPREHENSIONS — complete reference")
    print("=" * 70)
    demo_iteration_builtins()
    demo_type_conversion()
    demo_numeric_builtins()
    demo_introspection_builtins()
    demo_remaining_builtins()
    demo_loops()
    demo_comprehensions()
    demo_sorting_functional()
    demo_string_builtins()
    demo_io_builtins()
    print("-" * 70)
    print("All built-in/loop/comprehension demos passed ✔")


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()